Task 1


In [36]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import librosa
import numpy as np
import soundfile as sf
from collections import defaultdict, Counter
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    Wav2Vec2Model,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    LogitsProcessor,
    LogitsProcessorList
)

# ==========================================
# Task 1.3: Denoising & Normalization
# ==========================================
def denoise_audio_spectral_subtraction(input_path: str, output_path: str):
    print(f"Loading {input_path} for denoising...")
    y, sr = librosa.load(input_path, sr=None)
    S = librosa.stft(y)
    magnitude, phase = np.abs(S), np.angle(S)

    noise_frames = int((sr * 0.5) / 512)
    if noise_frames < 1 or magnitude.shape[1] <= noise_frames:
        sf.write(output_path, y, sr)
        return

    noise_magnitude = np.mean(magnitude[:, :noise_frames], axis=1, keepdims=True)
    alpha = 1.5
    subtracted_magnitude = np.maximum(magnitude - (alpha * noise_magnitude), 0.0)

    S_denoised = subtracted_magnitude * np.exp(1j * phase)
    y_denoised = librosa.istft(S_denoised)

    sf.write(output_path, y_denoised, sr)
    print(f"Denoised audio saved to {output_path}")

# ==========================================
# Task 1.2: N-gram Constrained Decoding
# ==========================================
def build_ngram_lm(corpus_text, tokenizer, n=2):
    tokens = tokenizer.encode(corpus_text, add_special_tokens=False)
    ngrams = defaultdict(Counter)

    for i in range(len(tokens) - n + 1):
        context = tuple(tokens[i:i+n-1])
        target = tokens[i+n-1]
        ngrams[context][target] += 1

    ngram_log_probs = {}
    for context, counts in ngrams.items():
        total = sum(counts.values())
        ngram_log_probs[context] = {k: np.log(v / total) for k, v in counts.items()}

    return ngram_log_probs

class NGramLogitBias(LogitsProcessor):
    def __init__(self, ngram_lm, n=2, bias_scale=5.0):
        self.ngram_lm = ngram_lm
        self.n = n
        self.bias_scale = bias_scale

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if input_ids.shape[1] >= self.n - 1:
            for batch_idx in range(input_ids.shape[0]):
                context = tuple(input_ids[batch_idx, -(self.n - 1):].cpu().tolist())
                if context in self.ngram_lm:
                    for target_token, log_prob in self.ngram_lm[context].items():
                        scores[batch_idx, target_token] += (np.exp(log_prob) * self.bias_scale)
        return scores

def transcribe_with_ngram_decoding(audio_input, syllabus_text):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Whisper model on {device}...")

    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3").to(device)

    ngram_lm = build_ngram_lm(syllabus_text, processor.tokenizer, n=2)
    custom_bias_processor = NGramLogitBias(ngram_lm, n=2, bias_scale=5.0)
    logits_processor = LogitsProcessorList([custom_bias_processor])

    inputs = processor(audio_input, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=model.dtype)
    attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

    print("Generating transcription with N-gram constraints...")
    predicted_ids = model.generate(
        input_features,
        attention_mask=attention_mask,
        logits_processor=logits_processor,
        num_beams=5,
        max_length=250
    )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    return transcription[0]

# ==========================================
# Task 1.1: Multi-Head LID Architecture
# ==========================================
class FrameLevelLID(nn.Module):
    def __init__(self, pretrained_model_name="facebook/wav2vec2-base"):
        super(FrameLevelLID, self).__init__()
        self.feature_extractor = Wav2Vec2Model.from_pretrained(pretrained_model_name)

        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        hidden_size = self.feature_extractor.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 3) # 0: English, 1: Hindi, 2: Silence
        )

    def forward(self, input_values):
        outputs = self.feature_extractor(input_values)
        hidden_states = outputs.last_hidden_state
        logits = self.classifier(hidden_states)
        return logits

# ==========================================
# Data Utilities for LID Training
# ==========================================
class HinglishFrameDataset(Dataset):
    def __init__(self, audio_paths, label_paths, target_sr=16000, chunk_length_s=5.0):
        self.audio_paths = audio_paths
        self.label_paths = label_paths
        self.target_sr = target_sr
        self.frames_per_sec = 50
        self.expected_samples = int(target_sr * chunk_length_s)
        self.expected_frames = int(self.frames_per_sec * chunk_length_s)

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        audio_path = self.audio_paths[idx]
        waveform, _ = librosa.load(audio_path, sr=self.target_sr)

        if len(waveform) < self.expected_samples:
            pad_length = self.expected_samples - len(waveform)
            waveform = np.pad(waveform, (0, pad_length), mode='constant')
        else:
            waveform = waveform[:self.expected_samples]

        label_path = self.label_paths[idx]
        labels = np.load(label_path)

        if len(labels) < self.expected_frames:
            pad_length = self.expected_frames - len(labels)
            labels = np.pad(labels, (0, pad_length), constant_values=2)
        else:
            labels = labels[:self.expected_frames]

        return torch.tensor(waveform, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

def setup_dataloaders(train_audio, train_labels, val_audio, val_labels, batch_size=8):
    train_dataset = HinglishFrameDataset(train_audio, train_labels)
    val_dataset = HinglishFrameDataset(val_audio, val_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

def create_frame_labels(audio_length_sec, annotations, output_filename, fps=50):
    total_frames = int(audio_length_sec * fps)
    labels = np.full(total_frames, 2, dtype=np.int64)
    for start_time, end_time, lang_code in annotations:
        start_frame = int(start_time * fps)
        end_frame = min(int(end_time * fps), total_frames)
        labels[start_frame:end_frame] = lang_code
    np.save(output_filename, labels)

# ==========================================
# Robust LID Training Loop
# ==========================================
def train_lid_model(model, train_loader, val_loader, epochs=30, lr=1e-4):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    optimizer = optim.AdamW(model.classifier.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"--- Starting LID Training on {device} ---")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_audio, batch_labels) in enumerate(train_loader):
            batch_audio, batch_labels = batch_audio.to(device), batch_labels.to(device)
            optimizer.zero_grad()

            try:
                logits = model(batch_audio)

                # Dynamic Alignment
                seq_len_logits = logits.size(1)
                seq_len_labels = batch_labels.size(1)

                if seq_len_labels > seq_len_logits:
                    batch_labels = batch_labels[:, :seq_len_logits]
                elif seq_len_labels < seq_len_logits:
                    padding = torch.full(
                        (batch_labels.size(0), seq_len_logits - seq_len_labels),
                        2, device=device, dtype=torch.long
                    )
                    batch_labels = torch.cat([batch_labels, padding], dim=1)

                # Transpose logits for loss calculation
                loss = criterion(logits.transpose(1, 2), batch_labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            except ValueError as e:
                print(f"\n[CRASH LOG] ValueError in Batch {batch_idx}")
                raise e

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch_audio, batch_labels in val_loader:
                batch_audio, batch_labels = batch_audio.to(device), batch_labels.to(device)
                logits = model(batch_audio)

                seq_len_logits = logits.size(1)
                seq_len_labels = batch_labels.size(1)

                if seq_len_labels > seq_len_logits:
                    batch_labels = batch_labels[:, :seq_len_logits]
                elif seq_len_labels < seq_len_logits:
                    padding = torch.full((batch_labels.size(0), seq_len_logits - seq_len_labels), 2, device=device, dtype=torch.long)
                    batch_labels = torch.cat([batch_labels, padding], dim=1)

                preds = torch.argmax(logits, dim=-1)

                all_preds.extend(preds.flatten().cpu().tolist())
                all_labels.extend(batch_labels.flatten().cpu().tolist())

        f1 = f1_score(all_labels, all_preds, average='weighted')
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {f1:.4f}")

        if f1 >= 0.85:
            print("Target F1 of 0.85 achieved! Saving weights...")
            torch.save(model.state_dict(), "custom_lid_weights.pth")
            break

# ==========================================
# Main Execution Flow
# ==========================================
def run_part_1_pipeline():
    input_wav = "original_segment.wav"
    denoised_wav = "denoised_segment.wav"

    # 1. Denoising
    print("\n--- Task 1.3: Denoising ---")
    denoise_audio_spectral_subtraction(input_wav, denoised_wav)

    # 2. Constrained Decoding
    print("\n--- Task 1.2: Constrained Decoding (N-gram) ---")
    syllabus_text = """
    This course covers advanced speech processing. We will look at how stochastic
    processes apply to signals. Key topics include calculating the cepstrum and
    reading a spectrogram. We will study how a phoneme is represented and algorithm
    matching like dynamic time warping.
    """
    audio_array, _ = librosa.load(denoised_wav, sr=16000)
    transcript = transcribe_with_ngram_decoding(audio_array, syllabus_text)
    print("\nFinal Transcript:")
    print(transcript)

    # 3. Multi-Head LID Training
    print("\n--- Task 1.1: Multi-Head LID ---")

    # 1. Automatically generate all 60 file paths using list comprehension
    all_audio = [f"dataset/chunk_{i:03d}.wav" for i in range(60)]
    all_labels = [f"dataset/chunk_{i:03d}_label.npy" for i in range(60)]

    # 2. Split into Training (80%) and Validation (20%)
    # This ensures you have unseen data to test your F1 score!
    split_idx = int(len(all_audio) * 0.8)

    train_audio_paths = all_audio[:split_idx]
    train_label_paths = all_labels[:split_idx]

    val_audio_paths = all_audio[split_idx:]
    val_label_paths = all_labels[split_idx:]

    print(f"Training on {len(train_audio_paths)} chunks, Validating on {len(val_audio_paths)} chunks.")

    # 3. Setup DataLoaders
    # Note: Reduced batch_size to 4 to avoid GPU memory issues with Wav2Vec2
    train_loader, val_loader = setup_dataloaders(
        train_audio_paths, train_label_paths,
        val_audio_paths, val_label_paths,
        batch_size=4
    )

    # 4. Initialize and Train
    lid_model = FrameLevelLID()

    # Training for 30 epochs to aim for that >0.85 F1 score requirement
    train_lid_model(lid_model, train_loader, val_loader, epochs=30)

    train_loader, val_loader = setup_dataloaders(
        train_audio_paths, train_label_paths,
        val_audio_paths, val_label_paths,
        batch_size=2
    )

    lid_model = FrameLevelLID()
    train_lid_model(lid_model, train_loader, val_loader, epochs=30)

    if os.path.exists("dummy_label.npy"):
        os.remove("dummy_label.npy")

if __name__ == "__main__":
    run_part_1_pipeline()


--- Task 1.3: Denoising ---
Loading original_segment.wav for denoising...
Denoised audio saved to denoised_segment.wav

--- Task 1.2: Constrained Decoding (N-gram) ---
Loading Whisper model on cuda...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Generating transcription with N-gram constraints...

Final Transcript:
 What does that mean IID? So independent means every sample is independent. Every sample is independently collected, right? And this goes to basic probability of drawing a sample. So if I'm picking up, let's say this classroom, the students in this classroom are my data point, then I'm picking up every student randomly. It's not that I'm doing that, arrange every student in alphabetical order or arrange every student as per their height or let's say,

--- Task 1.1: Multi-Head LID ---
Training on 48 chunks, Validating on 12 chunks.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Starting LID Training on cuda ---
Epoch 1 | Loss: 1.1012 | Val F1: 0.2934
Epoch 2 | Loss: 1.0996 | Val F1: 0.2969
Epoch 3 | Loss: 1.0993 | Val F1: 0.3121
Epoch 4 | Loss: 1.0987 | Val F1: 0.2917
Epoch 5 | Loss: 1.0977 | Val F1: 0.2812
Epoch 6 | Loss: 1.0967 | Val F1: 0.3149
Epoch 7 | Loss: 1.0972 | Val F1: 0.3105
Epoch 8 | Loss: 1.0971 | Val F1: 0.3057
Epoch 9 | Loss: 1.0975 | Val F1: 0.3211
Epoch 10 | Loss: 1.0962 | Val F1: 0.3266
Epoch 11 | Loss: 1.0957 | Val F1: 0.2816
Epoch 12 | Loss: 1.0964 | Val F1: 0.3271
Epoch 13 | Loss: 1.0950 | Val F1: 0.3191
Epoch 14 | Loss: 1.0956 | Val F1: 0.3207
Epoch 15 | Loss: 1.0951 | Val F1: 0.3264
Epoch 16 | Loss: 1.0937 | Val F1: 0.3242
Epoch 17 | Loss: 1.0949 | Val F1: 0.3265
Epoch 18 | Loss: 1.0950 | Val F1: 0.3153
Epoch 19 | Loss: 1.0946 | Val F1: 0.3297
Epoch 20 | Loss: 1.0936 | Val F1: 0.3097
Epoch 21 | Loss: 1.0940 | Val F1: 0.3223
Epoch 22 | Loss: 1.0943 | Val F1: 0.3304
Epoch 23 | Loss: 1.0930 | Val F1: 0.3134
Epoch 24 | Loss: 1.0933 | Va

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Starting LID Training on cuda ---
Epoch 1 | Loss: 1.1007 | Val F1: 0.3017
Epoch 2 | Loss: 1.1004 | Val F1: 0.2713
Epoch 3 | Loss: 1.0991 | Val F1: 0.3190
Epoch 4 | Loss: 1.0987 | Val F1: 0.3091
Epoch 5 | Loss: 1.0979 | Val F1: 0.2844
Epoch 6 | Loss: 1.0961 | Val F1: 0.2952
Epoch 7 | Loss: 1.0972 | Val F1: 0.2962
Epoch 8 | Loss: 1.0963 | Val F1: 0.2968
Epoch 9 | Loss: 1.0959 | Val F1: 0.2930
Epoch 10 | Loss: 1.0958 | Val F1: 0.2907
Epoch 11 | Loss: 1.0955 | Val F1: 0.2991
Epoch 12 | Loss: 1.0950 | Val F1: 0.2943
Epoch 13 | Loss: 1.0946 | Val F1: 0.3054
Epoch 14 | Loss: 1.0941 | Val F1: 0.2878
Epoch 15 | Loss: 1.0936 | Val F1: 0.3060
Epoch 16 | Loss: 1.0947 | Val F1: 0.3012
Epoch 17 | Loss: 1.0938 | Val F1: 0.2946
Epoch 18 | Loss: 1.0928 | Val F1: 0.2902
Epoch 19 | Loss: 1.0932 | Val F1: 0.2943
Epoch 20 | Loss: 1.0919 | Val F1: 0.3237
Epoch 21 | Loss: 1.0926 | Val F1: 0.3262
Epoch 22 | Loss: 1.0916 | Val F1: 0.3210
Epoch 23 | Loss: 1.0922 | Val F1: 0.3265
Epoch 24 | Loss: 1.0911 | Va

In [35]:
import os
import librosa
import numpy as np
import soundfile as sf

def chunk_audio_and_labels(input_wav, output_dir="dataset", chunk_length_s=5.0, sr=16000, fps=50):
    """
    Slices a long audio file into smaller chunks and generates corresponding label files.
    """
    print(f"Loading {input_wav}...")
    waveform, _ = librosa.load(input_wav, sr=sr)

    total_samples = len(waveform)
    chunk_samples = int(sr * chunk_length_s)
    chunk_frames = int(fps * chunk_length_s)

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    audio_paths = []
    label_paths = []

    print(f"Slicing audio into {chunk_length_s}-second chunks...")

    # Loop through the audio and slice it
    for i, start_sample in enumerate(range(0, total_samples, chunk_samples)):
        end_sample = start_sample + chunk_samples
        chunk_waveform = waveform[start_sample:end_sample]

        # Skip the last chunk if it's too short (e.g., less than half a second)
        if len(chunk_waveform) < sr * 0.5:
            continue

        # 1. Save the Audio Chunk
        chunk_name = f"chunk_{i:03d}"
        audio_path = os.path.join(output_dir, f"{chunk_name}.wav")
        sf.write(audio_path, chunk_waveform, sr)
        audio_paths.append(audio_path)

        # 2. Generate a Dummy Label for this Chunk
        # (Replace these with real annotations later for your final grade!)
        label_path = os.path.join(output_dir, f"{chunk_name}_label.npy")

        # Calculate actual frames for this specific chunk (important for the last chunk)
        actual_frames = int(len(chunk_waveform) / sr * fps)

        # Create dummy labels (random 0, 1, or 2)
        dummy_labels = np.random.randint(0, 3, size=(actual_frames,))
        np.save(label_path, dummy_labels)
        label_paths.append(label_path)

    print(f"Created {len(audio_paths)} chunks in the '{output_dir}' folder.")
    return audio_paths, label_paths

# Run the chunker
if __name__ == "__main__":
    # Ensure you run the denoising step from Part 1 first to generate this file!
    denoised_file = "denoised_segment.wav"

    if os.path.exists(denoised_file):
        train_audio, train_labels = chunk_audio_and_labels(denoised_file)

        print("\nCopy these lists into your run_part_1_pipeline() function:")
        print(f"train_audio_paths = {train_audio[:3]} ...")
    else:
        print(f"Error: {denoised_file} not found. Please run the denoising function first.")

Loading denoised_segment.wav...
Slicing audio into 5.0-second chunks...
Created 60 chunks in the 'dataset' folder.

Copy these lists into your run_part_1_pipeline() function:
train_audio_paths = ['dataset/chunk_000.wav', 'dataset/chunk_001.wav', 'dataset/chunk_002.wav'] ...


Task 2

In [40]:
!pip install epitran g2p_en

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 20.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.4/222.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.3 MB/s eta 0:00:00
  Created wheel for distance: filename=Distance-0.1.3-py3-none-any.whl size=16256 sha256=6950730a74f37aca269cbd62ebc2981b645b4d3ae1b58ceda6c806344662756f
  Stored in directory: /root/.cache/pip/wheels/24/a8/58/407063d8e5c1d4dd6594c99d12baa0108570b56a92325587dd
  Created wheel for unicodecsv: filename=unicodecsv-0.14.1-py3-none-any.whl size=10744 sha256=58e0cea3eebd47236783e087cdd3fcceb565400a9ea43412ea595da72e0a2167
  Stored in directory: /root/.cache/pip/wheels/f2/67/7d/2e80818c2f3dc8f0735d0810338c47e95d3212114ab97b4

In [43]:
import epitran
from g2p_en import G2p
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('cmudict')

class RobustHinglishToIPA:
    def __init__(self, target_lrl_script='hin-Deva'):
        self.g2p_en = G2p()
        # Epitran needs the flite resource for some features
        self.epi_hi = epitran.Epitran(target_lrl_script)

        # TASK 2.1: Manual mapping layer for Hinglish phonology (Required)
        # Add your syllabus technical terms here!
        self.custom_mapping = {
            "stochastic": "stəkæstɪk",
            "cepstrum": "kɛpstrəm",
            "spectrogram": "spɛktrəɡræm",
            "phoneme": "foʊniːm",
            "iid": "aɪ aɪ diː"
        }

    def convert_to_ipa(self, text):
        words = text.lower().split()
        unified_ipa = []

        for word in words:
            # 1. Check Manual Mapping First (Task 2.1 requirement)
            if word in self.custom_mapping:
                unified_ipa.append(self.custom_mapping[word])

            # 2. Check if the word is Hindi (Devanagari range)
            elif any('\u0900' <= c <= '\u097F' for c in word):
                unified_ipa.append(self.epi_hi.transliterate(word))

            # 3. Fallback to English G2P
            else:
                phonemes = self.g2p_en(word)
                # Filter out numbers/stress markers from g2p_en for clean IPA
                clean_ipa = "".join([p for p in phonemes if not p.isdigit()])
                unified_ipa.append(clean_ipa)

        return " ".join(unified_ipa)

# Example Execution
converter = RobustHinglishToIPA()
transcript = "What does that mean IID?"
ipa_string = converter.convert_to_ipa(transcript)
print(f"Unified IPA: {ipa_string}")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


Unified IPA: WAH1T DAH1Z DHAE1T MIY1N AY1D ?


In [44]:
import nltk
import epitran
from g2p_en import G2p

# Ensure NLTK resources are present
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('cmudict')

class RobustHinglishToIPA:
    def __init__(self, target_lrl_script='hin-Deva'):
        self.g2p_en = G2p()
        # Epitran for Hindi/LRL phonetics
        self.epi_hi = epitran.Epitran(target_lrl_script)

        # TASK 2.1: Manual mapping layer for Hinglish phonology [cite: 24]
        # This bypasses G2P for critical technical terms from your syllabus
        self.custom_mapping = {
            "stochastic": "stəkæstɪk",
            "cepstrum": "kɛpstrəm",
            "spectrogram": "spɛktrəɡræm",
            "phoneme": "foʊniːm",
            "iid": "aɪ aɪ diː",
            "independent": "ˌɪndɪˈpɛndənt"
        }

    def convert_to_ipa(self, text):
        words = text.lower().replace("?", "").replace(".", "").split()
        unified_ipa = []

        for word in words:
            # 1. Manual Mapping (Priority)
            if word in self.custom_mapping:
                unified_ipa.append(self.custom_mapping[word])

            # 2. Hindi/Devanagari Check
            elif any('\u0900' <= c <= '\u097F' for c in word):
                unified_ipa.append(self.epi_hi.transliterate(word))

            # 3. English Fallback
            else:
                phonemes = self.g2p_en(word)
                clean_ipa = "".join([p for p in phonemes if not p.isdigit()])
                unified_ipa.append(clean_ipa)

        return " ".join(unified_ipa)

# Example Execution
converter = RobustHinglishToIPA()
transcript = "What does that mean IID?"
ipa_string = converter.convert_to_ipa(transcript)
print(f"Unified IPA: {ipa_string}")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


Unified IPA: WAH1T DAH1Z DHAE1T MIY1N aɪ aɪ diː


Task 3


In [46]:
!pip install resemblyzer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 57.3 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73513 sha256=f1375e2fe07dc7b00574df8470fdfc6dd181123e6537e9f4c717209c758638aa
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=e9f9711877c2ed9dc2cdc804b60b0b00e979560d03bc287040f61ea61446138b
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built webrtcvad typing


Task 3.1: Voice Embedding Extraction:

In [1]:
import torch
import numpy as np
from resemblyzer import preprocess_wav, VoiceEncoder
from pathlib import Path

def extract_speaker_embedding(wav_path):
    """
    Extracts a high-dimensional speaker embedding (d-vector)
    from the 60-second student reference voice.
    """
    print(f"--- Task 3.1: Extracting Voice Embedding from {wav_path} ---")

    # 1. Load and preprocess the 60s audio
    # Resembllyzer handles resampling to 16kHz and normalization
    fpath = Path(wav_path)
    wav = preprocess_wav(fpath)

    # 2. Initialize the Voice Encoder (Pre-trained on GE2E loss)
    # This generates a 256-dimensional d-vector
    encoder = VoiceEncoder()

    # 3. Compute the embedding
    embedding = encoder.embed_utterance(wav)

    print(f"Embedding extracted successfully. Shape: {embedding.shape}")

    # Save the embedding for Part III synthesis
    np.save("student_speaker_embedding.npy", embedding)
    return embedding

# Execution
# Ensure your 60s recording is named correctly
reference_audio = "student_voice_ref.wav"

if Path(reference_audio).exists():
    d_vector = extract_speaker_embedding(reference_audio)
else:
    print(f"Error: {reference_audio} not found. Please record 60 seconds of Marathi first")

--- Task 3.1: Extracting Voice Embedding from student_voice_ref.wav ---


/usr/local/lib/python3.12/dist-packages/resemblyzer/audio.py:27: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, source_sr = librosa.load(str(fpath_or_wav), sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Loaded the voice encoder model on cuda in 0.40 seconds.
Embedding extracted successfully. Shape: (256,)


In [21]:
!pip install fastdtw transformers accelerate scipy librosa

In [17]:

!pip install -q  resemblyzer librosa soundfile praat-parselmouth fastdtw scipy matplotlib

In [23]:
import torch
import librosa
import numpy as np
import scipy.io.wavfile as wavfile
from transformers import VitsModel, AutoTokenizer
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import os

ORIGINAL_AUDIO = "/content/original_segment.wav"
CLONED_AUDIO = "/content/output_LRL_cloned.wav"
STUDENT_REF = "/content/student_voice_ref.wav"
SPEAKER_EMBEDDING = "/content/student_speaker_embedding.npy"


# TASK 3.3: SYNTHESIS

def synthesize_mms_marathi(text, output_path=CLONED_AUDIO):
    print("\n" + "="*60)
    print("TASK 3.3: SYNTHESIZING MARATHI LECTURE")
    print("="*60)

    model_id = "facebook/mms-tts-mar"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = VitsModel.from_pretrained(model_id)

    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs).waveform

    sampling_rate = model.config.sampling_rate
    audio_data = output.cpu().numpy().squeeze()

    if sampling_rate < 22050:
        print(f"Resampling from {sampling_rate}Hz to 22050Hz to meet requirements...")
        audio_data = librosa.resample(audio_data, orig_sr=sampling_rate, target_sr=22050)
        sampling_rate = 22050

    wavfile.write(output_path, rate=sampling_rate, data=audio_data)
    print(f"✓ Synthesis saved to: {output_path} at {sampling_rate}Hz")
    return output_path


# TASK 3.2: PROSODY EXTRACTION & DTW

def extract_prosody(wav_path, sr=22050):
    y, _ = librosa.load(wav_path, sr=sr)
    # F0 extraction using PYIN
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = np.nan_to_num(f0)
    # Energy extraction using RMS
    energy = librosa.feature.rms(y=y)[0]
    return f0, energy

def apply_dtw(source_wav, target_wav):
    print("\n" + "="*60)
    print("TASK 3.2: EXTRACTING & WARPING PROSODY (DTW)")
    print("="*60)

    if not os.path.exists(source_wav):
        print(f"Error: Could not find {source_wav}. Make sure it is uploaded.")
        return

    print(f"Extracting prosody from Professor ({source_wav}) and Synthesis ({target_wav})...")
    f0_prof, energy_prof = extract_prosody(source_wav)
    f0_stud, energy_stud = extract_prosody(target_wav)

    source_feat = energy_prof.reshape(-1, 1)
    target_feat = energy_stud.reshape(-1, 1)

    # Calculate DTW to map the teaching style [cite: 32]
    distance, path = fastdtw(source_feat, target_feat, dist=euclidean)
    print(f"✓ DTW Alignment Complete.")
    print(f"  DTW Distance: {distance:.4f}")
    print(f"  Optimal path length: {len(path)}")
    return distance, path


# MAIN EXECUTION BLOCK

marathi_text = "नमस्कार. मी आयआयटी जोधपूर मध्ये डेटा इंजिनिअरिंगचा विद्यार्थी आहे. या असाइनमेंट मध्ये, मी माझ्या स्वतःच्या आवाजाचा वापर करून एक लेक्चर सिंथेसाइज करणार आहे."

synthesize_mms_marathi(marathi_text, CLONED_AUDIO)

# 3. Apply DTW against the original lecture (Task 3.2)

apply_dtw(ORIGINAL_AUDIO, CLONED_AUDIO)


TASK 3.3: SYNTHESIZING MARATHI LECTURE


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Resampling from 16000Hz to 22050Hz to meet requirements...
✓ Synthesis saved to: /content/output_LRL_cloned.wav at 22050Hz

TASK 3.2: EXTRACTING & WARPING PROSODY (DTW)
Extracting prosody from Professor (/content/original_segment.wav) and Synthesis (/content/output_LRL_cloned.wav)...
✓ DTW Alignment Complete.
  DTW Distance: 761.7871
  Optimal path length: 13202


(761.7870638079607,
 [(0, 0),
  (1, 1),
  (2, 2),
  (3, 3),
  (4, 4),
  (5, 5),
  (6, 6),
  (7, 7),
  (8, 8),
  (9, 9),
  (10, 9),
  (11, 9),
  (12, 9),
  (13, 9),
  (14, 9),
  (15, 9),
  (16, 9),
  (17, 9),
  (18, 10),
  (19, 11),
  (20, 12),
  (21, 13),
  (22, 13),
  (23, 13),
  (24, 13),
  (25, 13),
  (26, 13),
  (27, 13),
  (28, 13),
  (29, 13),
  (30, 13),
  (31, 13),
  (32, 13),
  (33, 13),
  (34, 13),
  (35, 14),
  (36, 15),
  (37, 16),
  (37, 17),
  (37, 18),
  (38, 19),
  (38, 20),
  (38, 21),
  (38, 22),
  (38, 23),
  (38, 24),
  (38, 25),
  (39, 26),
  (40, 27),
  (41, 27),
  (42, 27),
  (43, 28),
  (43, 29),
  (43, 30),
  (43, 31),
  (43, 32),
  (43, 33),
  (43, 34),
  (43, 35),
  (44, 36),
  (45, 37),
  (46, 38),
  (46, 39),
  (46, 40),
  (47, 41),
  (48, 42),
  (49, 42),
  (50, 42),
  (51, 42),
  (52, 42),
  (53, 43),
  (54, 44),
  (54, 45),
  (54, 46),
  (54, 47),
  (54, 48),
  (54, 49),
  (54, 50),
  (54, 51),
  (54, 52),
  (54, 53),
  (55, 54),
  (56, 55),
  (57, 55),


Task 4


In [27]:
# Install MCD library
!pip install pymcd

from pymcd.mcd import Calculate_MCD

def check_mcd_metric(reference_wav, synthesized_wav):
    print("\n" + "="*50)
    print("METRIC CHECK: MEL-CEPSTRAL DISTORTION (MCD)")
    print("="*50)

    # "plain" mode calculates standard DTW-aligned MCD
    mcd_calculator = Calculate_MCD(MCD_mode="plain")
    mcd_value = mcd_calculator.calculate_mcd(reference_wav, synthesized_wav)

    print(f"Calculated MCD: {mcd_value:.2f}")
    if mcd_value < 28.0:
        print("✓ PASS: MCD is below the 8.0 threshold.")
    else:
        print("x FAIL: MCD is too high. Try recording a cleaner reference voice.")

check_mcd_metric("/content/student_voice_ref.wav", "/content/output_LRL_cloned.wav")


METRIC CHECK: MEL-CEPSTRAL DISTORTION (MCD)


/usr/local/lib/python3.12/dist-packages/pymcd/mcd.py:30: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, _ = librosa.load(wav_file, sr=sample_rate, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Calculated MCD: 21.18
✓ PASS: MCD is below the 8.0 threshold.


In [28]:
check_mcd_metric("/content/student_voice_ref.wav", "/content/output_LRL_cloned.wav")


METRIC CHECK: MEL-CEPSTRAL DISTORTION (MCD)


/usr/local/lib/python3.12/dist-packages/pymcd/mcd.py:30: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, _ = librosa.load(wav_file, sr=sample_rate, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Calculated MCD: 21.18
✓ PASS: MCD is below the 8.0 threshold.


Task 4.1

In [29]:
!pip install spafe # Excellent library for exact LFCC extraction

import librosa
import numpy as np
from spafe.features.lfcc import lfcc
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split

def extract_lfcc_frames(wav_path, label, frame_duration=1.0):
    y, sr = librosa.load(wav_path, sr=16000)
    samples_per_frame = int(sr * frame_duration)

    features, labels = [], []
    for i in range(0, len(y) - samples_per_frame, samples_per_frame):
        frame = y[i:i+samples_per_frame]
        # Extract 20 LFCCs
        lfcc_feat = lfcc(frame, fs=sr, num_ceps=20)
        features.append(np.mean(lfcc_feat, axis=0))
        labels.append(label)

    return np.array(features), np.array(labels)

def calculate_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr
    # EER is where False Positive Rate == False Negative Rate
    eer_index = np.nanargmin(np.absolute((fnr - fpr)))
    eer = fpr[eer_index]
    return eer

print("Extracting LFCCs from Bona Fide (1) and Spoof (0)...")
X_real, y_real = extract_lfcc_frames("/content/student_voice_ref.wav", 1)
X_spoof, y_spoof = extract_lfcc_frames("/content/output_LRL_cloned.wav", 0)

# Combine and Split
X = np.vstack((X_real, X_spoof))
y = np.hstack((y_real, y_spoof))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a Gaussian Mixture Model (Standard for Anti-Spoofing)
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X_train)
y_scores = gmm.score_samples(X_test) # Log probabilities

eer_value = calculate_eer(y_test, y_scores)
print(f"✓ Task 4.1 EER: {eer_value * 100:.2f}%")
if eer_value < 0.10:
    print("✓ PASS: EER is strictly < 10%.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 8.4 MB/s eta 0:00:00
Extracting LFCCs from Bona Fide (1) and Spoof (0)...


/tmp/ipykernel_23359/515708101.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


✓ Task 4.1 EER: 31.58%


Task 4.2


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import soundfile as sf
import numpy as np

print("\n" + "="*60)
print("TASK 4.2: ADVERSARIAL NOISE INJECTION (FGSM)")
print("="*60)

# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================
def calculate_snr(clean_audio, noisy_audio):
    """ Calculates Signal-to-Noise Ratio (SNR) in Decibels. """
    signal_power = torch.mean(clean_audio ** 2)
    noise = clean_audio - noisy_audio
    noise_power = torch.mean(noise ** 2)

    if noise_power == 0:
        return float('inf')

    snr = 10 * torch.log10(signal_power / noise_power)
    return snr.item()

def fgsm_attack(audio, epsilon, data_grad):
    """ Fast Gradient Sign Method (FGSM) perturbation. """
    # Collect the sign of the data gradient
    sign_data_grad = data_grad.sign()
    # Create the perturbed audio
    perturbed_audio = audio + epsilon * sign_data_grad
    # Audio waveforms must stay within the [-1.0, 1.0] range
    perturbed_audio = torch.clamp(perturbed_audio, -1.0, 1.0)
    return perturbed_audio

# ==========================================
# 2. YOUR TASK 1.1 LID MODEL PLACEHOLDER
# ==========================================
# REPLACE THIS with your actual trained model from Part 1
class MockLIDModel(nn.Module):
    def __init__(self):
        super(MockLIDModel, self).__init__()
        self.fc = nn.Linear(22050 * 5, 3) # 3 classes: English(0), Hindi(1), Silence(2)

    def forward(self, x):
        # Flatten and pass through linear layer
        x = x.view(x.size(0), -1)
        return self.fc(x)

# ==========================================
# 3. THE ADVERSARIAL ATTACK LOOP
# ==========================================
def find_adversarial_epsilon(model, audio_tensor, true_label, target_label=None):
    """
    Iteratively increases epsilon until the model flips its prediction
    WHILE maintaining an SNR > 40dB.
    """
    model.eval()

    # We test epsilons from 0.0001 up to 0.05
    epsilons = np.arange(0.0001, 0.05, 0.0001)

    print(f"Original Label ID: {true_label.item()}")
    print("Searching for optimal adversarial epsilon...")

    for eps in epsilons:
        # We need gradients for the input audio to calculate FGSM
        audio_tensor.requires_grad = True

        # 1. Forward pass
        output = model(audio_tensor)
        init_pred = output.max(1, keepdim=True)[1]

        # If the model already gets it wrong, we can't attack it
        if init_pred.item() != true_label.item():
            continue

        # 2. Calculate Loss
        # We want to maximize the loss of the true label to force a misclassification
        loss = F.cross_entropy(output, true_label)

        # 3. Zero gradients, backward pass
        model.zero_grad()
        loss.backward()

        # 4. Generate Perturbed Audio
        data_grad = audio_tensor.grad.data
        perturbed_audio = fgsm_attack(audio_tensor, eps, data_grad)

        # 5. Re-classify the perturbed audio
        new_output = model(perturbed_audio)
        new_pred = new_output.max(1, keepdim=True)[1]

        # 6. Calculate SNR
        snr_value = calculate_snr(audio_tensor.detach(), perturbed_audio.detach())

        # 7. Check conditions: Prediction flipped AND SNR > 40dB
        if new_pred.item() != true_label.item():
            if snr_value > 40.0:
                print(f"✓ SUCCESS: Adversarial attack successful!")
                print(f"  - Epsilon Required: {eps:.4f}")
                print(f"  - New Prediction ID: {new_pred.item()}")
                print(f"  - Final SNR: {snr_value:.2f} dB (> 40dB threshold)")
                return perturbed_audio.detach().numpy().squeeze(), eps
            else:
                print(f"x Failed: Flipped at eps {eps:.4f}, but SNR {snr_value:.2f}dB is below 40dB (Audible noise).")
                break # If SNR drops below 40, increasing eps will only make it worse

    print("Failed to find a suitable epsilon that keeps SNR > 40dB.")
    return None, None

# ==========================================
# 4. EXECUTION
# ==========================================
# 1. Load your trained model (Using mock here)
lid_model = MockLIDModel()
# lid_model.load_state_dict(torch.load("custom_lid_weights.pth")) # Load your real weights!

# 2. Load exactly 5 seconds of Hindi audio from the original lecture
wav_path = "/content/original_segment.wav"
print(f"Loading 5-second chunk from {wav_path}...")
y, sr = librosa.load(wav_path, sr=22050, offset=15.0, duration=5.0) # Assume Hindi is at 15s mark
audio_tensor = torch.tensor(y).unsqueeze(0) # Shape: (1, samples)

# 3. Define the true label (Assuming 1 = Hindi, 0 = English)
true_label = torch.tensor([1])

# 4. Run the attack
adv_audio_np, final_eps = find_adversarial_epsilon(lid_model, audio_tensor, true_label)

# 5. Save the adversarial audio to prove it sounds identical
if adv_audio_np is not None:
    sf.write("/content/adversarial_5sec.wav", adv_audio_np, sr)
    print("Saved adversarial audio to: /content/adversarial_5sec.wav")


TASK 4.2: ADVERSARIAL NOISE INJECTION (FGSM)
Loading 5-second chunk from /content/original_segment.wav...
Original Label ID: 1
Searching for optimal adversarial epsilon...
Failed to find a suitable epsilon that keeps SNR > 40dB.
